# Bab 11 — Neural networks that understand language: king – man + woman == ?

**Mahasiswa:** Tarannia Raisha Putri · 101032300231 · TK4702 · Deep Learning

**Referensi utama:** Andrew W. Trask, *Grokking Deep Learning*, Manning (2019), Bab 11, halaman cetak 187–208 (PDF 208–229, indeks PDF mulai 1). Subbab spesifik dicantumkan pada setiap tahap.

**Tujuan pembelajaran:** Mereproduksi bag-of-words, sentiment embeddings, tetangga kata, CBOW dengan negative sampling, dan analogi vektor.

## Ringkasan

Teks diubah menjadi indeks kata dan embedding. Loss sentiment mengelompokkan kata berdasarkan polaritas; loss menebak kata yang hilang mempelajari pola konteks yang berbeda. Makna vektor bergantung pada tugas, bukan label semantik yang ditanam ke setiap neuron.

## Konsep, intuisi, dan notasi

Vektor multi-hot $x\in\{0,1\}^V$ memilih baris $E\in\mathbb R^{V\times H}$: $xE=\sum_{w\in\mathrm{review}}E_w$. Embedding lookup menghindari perkalian banyak nol. Contoh sentiment: $h=\sigma(\sum E_w)$, $p=\sigma(hW)$, $W\in\mathbb R^{H\times1}$. Untuk binary cross-entropy $d_z=p-y$; gradien ke embedding perlu faktor $h(1-h)$.

CBOW memakai rata-rata embedding konteks $h=\frac1{|C|}\sum_{w\in C}E_w$. Untuk satu kata positif dan lima negatif, $p=\sigma(O_Sh)$, $O_S\in\mathbb R^{6\times H}$, $L=\sum_j[\log(1+e^{z_j})-y_jz_j]$. Gradien setiap embedding konteks adalah $(O_S^T(p-y))/|C|$.

Kata yang muncul dalam konteks mirip dapat dekat meski sentimennya berlawanan. Dimensi tunggal dapat berotasi tanpa mengubah relasi yang relevan; jangan menamai satu neuron sebagai konsep universal. Analogi $e_a-e_b+e_c$ adalah pencarian geometris, bukan jaminan penalaran logis.

## Persiapan dan cara membaca

Jalankan **Restart Kernel → Run All**. Semua penjelasan merupakan parafrasa orisinal; angka hasil berada pada output sel yang benar-benar dijalankan. Label **reproduksi** berarti algoritme/contoh buku ditulis ulang; **koreksi/penyesuaian** menunjukkan perubahan; **tambahan** menunjukkan eksperimen atau interpretasi praktikum. `GDL_MODE=quick` mengurangi latihan berat; `full` memilih anggaran buku, dengan perubahan yang tetap dicatat. Keberhasilan eksekusi tidak membuktikan kesamaan akurasi dengan buku.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Jalankan dari root repository atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = os.environ.get('GDL_MODE', 'quick')
assert MODE in {'quick', 'full'}
FULL = MODE == 'full'
np.random.seed(1)
np.set_printoptions(precision=6, suppress=True)
METRICS = {}
def record(**values):
    METRICS.update(values)
def finite(*arrays):
    assert all(np.isfinite(np.asarray(a)).all() for a in arrays), 'NaN/inf terdeteksi'
def softmax(z):
    z = np.asarray(z)
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))
print('Mode:', MODE, '| NumPy:', np.__version__)

Mode: quick | NumPy: 2.5.4


<a id="b11-bow"></a>

## Reproduksi: NLP, IMDB, encoding, dan embedding

**Sumber:** cetak 188–194; PDF 209–215.

`reviews.txt` dan `labels.txt` pendamping berisi 25.000 pasangan binary, berbeda dari uraian informal rating bintang pada buku. 24.000 awal merupakan train, 1.000 akhir evaluasi seperti listing. Vocabulary dibuat dari train saja untuk menghindari informasi kosa kata evaluasi; sorted vocabulary menggantikan urutan set yang tidak deterministik. Kata tak dikenal diabaikan dan dihitung; tidak ada dataset sintetis pengganti.

In [2]:
from src.datasets import reviews
raw_reviews,targets=reviews()
onehots=dict(zip(['cat','the','dog','sat'],np.eye(4,dtype=int)))
print('the cat sat:',sum(onehots[w] for w in ['the','cat','sat']))
assert np.array_equal(sum(onehots[w] for w in ['the','cat','sat']),[1,1,0,1])
vocabulary=sorted({w for line in raw_reviews[:24000] for w in line.split()})
w2i={w:i for i,w in enumerate(vocabulary)}
dataset=[np.array(sorted({w2i[w] for w in line.split() if w in w2i}),dtype=int) for line in raw_reviews]
N=24000 if FULL else 1200
print('Review total:',len(dataset),'vocab training:',len(vocabulary),'train dipakai:',N,'test:',1000)
print('OOV test occurrences:',sum(w not in w2i for line in raw_reviews[24000:] for w in line.split()))
E_demo=np.arange(12).reshape(4,3)
np.testing.assert_allclose(np.array([1,1,0,1])@E_demo,E_demo[[0,1,3]].sum(0))

the cat sat: [1 1 0 1]


Review total: 25000 vocab training: 72809 train dipakai: 1200 test: 1000
OOV test occurrences: 1838


<a id="b11-sentiment"></a>

## Reproduksi: sentiment training dan interpretasi

**Sumber:** cetak 195–200; PDF 216–221.

Dua jalur memperlihatkan koreksi halaman 195: listing melewatkan derivative sigmoid hidden. Versi buku dipertahankan sebagai perbandingan; versi koreksi memasukkan h(1−h). Satu review memakai himpunan kata unik. Dua epoch, hidden 100, alpha 0,01 mengikuti buku; mode ringan membatasi contoh yang dilatih.

In [3]:
def train_sentiment(corrected=True):
    rng=np.random.RandomState(1)
    E=.2*rng.rand(len(vocabulary),100)-.1; W=.2*rng.rand(100,1)-.1
    log=[]
    for epoch in range(2):
        total=0.; correct=0
        for ids,y in zip(dataset[:N],targets[:N]):
            h=sigmoid(E[ids].sum(0)); z=(h@W).item(); p=sigmoid(z)
            total+=np.logaddexp(0,z)-y*z; correct+=int((p>.5)==y)
            delta=p-y; dh=(W[:,0]*delta)
            if corrected: dh*=h*(1-h)
            E[ids]-=.01*dh
            W-=.01*np.outer(h,[delta])
        log.append([total/N,correct/N])
    probs=np.array([sigmoid((sigmoid(E[ids].sum(0))@W).item()) for ids in dataset[24000:]])
    accuracy=float(np.mean((probs>.5)==targets[24000:]))
    finite(E,W,log,probs)
    return E,W,log,accuracy
sent_book=train_sentiment(False); sent_fixed=train_sentiment(True)
for name,result in [('listing',sent_book),('koreksi',sent_fixed)]:
    print(name,'epoch [BCE,online accuracy]:',result[2],'test accuracy:',result[3])
E_sent=sent_fixed[0]
for word in ['beautiful','terrible']:
    distances=np.linalg.norm(E_sent-E_sent[w2i[word]],axis=1)
    print(word,[(vocabulary[i],round(float(distances[i]),4)) for i in distances.argsort()[:6]])
print('Tetangga aktual dapat masih didominasi inisialisasi pada latihan pendek.')

listing epoch [BCE,online accuracy]: [[np.float64(0.6640240806599981), 0.5708333333333333], [np.float64(0.42214273584965656), 0.8558333333333333]] test accuracy: 0.77
koreksi epoch [BCE,online accuracy]: [[np.float64(0.7066145818239049), 0.3875], [np.float64(0.6253028373437649), 0.72]] test accuracy: 0.699
beautiful [('beautiful', 0.0), ('gremlins', 0.5962), ('everone', 0.632), ('redubbed', 0.6499), ('dandy', 0.6512), ('chulawasse', 0.6596)]
terrible [('terrible', 0.0), ('grape', 0.6305), ('induni', 0.6385), ('anyways', 0.6552), ('prototypes', 0.6555), ('snatch', 0.6559)]
Tetangga aktual dapat masih didominasi inisialisasi pada latihan pendek.


<a id="b11-cbow"></a>

## Reproduksi: filling in the blank dan negative sampling

**Sumber:** cetak 201–205; PDF 222–226.

Window=2 pada narasi berarti dua kata kiri/kanan, tetapi slice listing mengambil hanya satu kata kanan. Koreksi memakai ujung t+3, membagi gradien konteks dengan jumlah kata, dan `np.add.at` untuk indeks yang berulang. Listing tetap dapat dijalankan melalui `corrected=False`. Output dua metode dicetak.

Vocabulary CBOW berasal dari corpus tanpa label, termasuk review evaluasi sentiment seperti alur buku; model CBOW ini **tidak dievaluasi sebagai classifier sentiment**. Mode ringan memakai 200 review teracak, masing-masing maksimal 80 token, dua lintasan; mode penuh memakai seluruh corpus, tanpa pemotongan.

In [4]:
def train_cbow(raw_reviews, full=False, corrected=True):
    import random
    rng=np.random.RandomState(1)
    tokens=[line.split() for line in raw_reviews]
    vocabulary=sorted({word for sent in tokens for word in sent})
    lookup={word:i for i,word in enumerate(vocabulary)}
    sequences=[[lookup[w] for w in sent] for sent in tokens]
    concatenated=np.array([i for sent in sequences for i in sent])
    random.Random(1).shuffle(sequences)
    selected=sequences if full else [sent[:80] for sent in sequences[:200]]
    E=(rng.rand(len(vocabulary),50)-.5)*.2; O=np.zeros_like(E)
    target=np.array([1.,0.,0.,0.,0.,0.]); losses=[]; updates=0
    for epoch in range(2):
        total=0.; count=0
        for sequence in selected:
            for t,word in enumerate(sequence):
                end=t+3 if corrected else t+2
                context=sequence[max(0,t-2):t]+sequence[t+1:min(len(sequence),end)]
                if not context: continue
                samples=[word]+concatenated[(rng.rand(5)*len(concatenated)).astype(int)].tolist()
                h=E[context].mean(0); logits=h@O[samples].T; p=sigmoid(logits)
                total+=np.sum(np.logaddexp(0,logits)-target*logits); count+=6
                d=p-target; dh=d@O[samples]; dO=np.outer(d,h)
                if corrected:
                    np.add.at(E,context,-.05*np.broadcast_to(dh/len(context),(len(context),50)))
                    np.add.at(O,samples,-.05*dO)
                else:
                    E[context]-=.05*dh
                    O[samples]-=.05*dO
                updates+=1
        losses.append(total/count)
    finite(E,O,losses)
    return E,O,vocabulary,lookup,losses,updates

def nearest(E,vocabulary,lookup,word,k=6):
    if word not in lookup: return [('OOV: '+word,0.)]
    distances=np.linalg.norm(E-E[lookup[word]],axis=1)
    return [(vocabulary[i],float(distances[i])) for i in np.argsort(distances)[:k]]

In [5]:
cbow_book=train_cbow(raw_reviews,FULL,False)
cbow_fixed=train_cbow(raw_reviews,FULL,True)
E,O,cbow_vocab,cbow_index,cbow_losses,updates=cbow_fixed
print('Listing CBOW: BCE rata-rata',cbow_book[4],'updates',cbow_book[5])
print('Koreksi CBOW: BCE rata-rata',cbow_losses,'updates',updates)
for word in ['beautiful','terrible']: print(word,nearest(E,cbow_vocab,cbow_index,word))

Listing CBOW: BCE rata-rata [np.float64(0.5718606214600217), np.float64(0.49402786794237147)] updates 31600
Koreksi CBOW: BCE rata-rata [np.float64(0.5932460807357012), np.float64(0.5147771602015686)] updates 31600
beautiful [('beautiful', 0.0), ('ricochet', 0.4124830062153853), ('gothas', 0.44315953662853225), ('switcheroo', 0.44754758816982887), ('fillum', 0.4508607218461159), ('adriensen', 0.45104265580408653)]
terrible [('terrible', 0.0), ('schenck', 0.3888286226348506), ('chalon', 0.3900996907938495), ('camadrie', 0.3910142628731705), ('planes', 0.3936896451701474), ('freeman', 0.39578535179156005)]


<a id="b11-analogy"></a>

## Reproduksi: king − man + woman dan word analogies

**Sumber:** cetak 206–208; PDF 227–229.

Untuk menjaga keterlacakan, `book=True` memakai perkalian squared norm seperti kode pendamping; itu bukan normalisasi L2. Versi koreksi membagi dengan norm dan mencari cosine similarity, serta mengecualikan kata kueri secara eksplisit. Hasil tidak dipaksakan menjadi queen; corpus dan pelatihan pendek mungkin tidak mendukung relasi tersebut.

In [6]:
def analogy(positive,negative,book=False):
    missing=[w for w in positive+negative if w not in cbow_index]
    if missing: return {'OOV':missing}
    norms=np.linalg.norm(E,axis=1,keepdims=True)
    transformed=E*norms**2 if book else E/np.maximum(norms,1e-12)
    query=sum(transformed[cbow_index[w]] for w in positive)-sum(transformed[cbow_index[w]] for w in negative)
    scores=-np.linalg.norm(E-query,axis=1) if book else transformed@query/max(np.linalg.norm(query),1e-12)
    skip=set(positive+negative)
    return [(cbow_vocab[i],float(scores[i])) for i in np.argsort(-scores) if cbow_vocab[i] not in skip][:5]
for plus,minus in [(['terrible','good'],['bad']),(['elizabeth','he'],['she']),(['king','woman'],['man'])]:
    print(plus,'minus',minus,'listing:',analogy(plus,minus,True),'koreksi:',analogy(plus,minus))
record(sentiment_train_count=N,sentiment_test_count=1000,book_sentiment_accuracy=sent_book[3],corrected_sentiment_accuracy=sent_fixed[3],cbow_updates=updates,cbow_final_bce=float(cbow_losses[-1]))

['terrible', 'good'] minus ['bad'] listing: [('being', -0.9151063934919137), ('into', -0.915499466780528), ('even', -0.9229721402878549), ('such', -0.9292400189295136), ('few', -0.9358841018895474)] koreksi: [('freeman', 0.5359206136585879), ('chalon', 0.5307405461079604), ('moira', 0.5206868366914359), ('darker', 0.5159385826360235), ('camadrie', 0.5126626069149404)]
['elizabeth', 'he'] minus ['she'] listing: [('his', -1.3554638349837225), ('who', -1.5411759793075945), ('that', -1.5544119792129714), ('an', -1.598896972427753), ('are', -1.6519091714187784)] koreksi: [('piracy', 0.5653921917052871), ('caballeros', 0.5489237554447941), ('fluff', 0.5058935555831702), ('shins', 0.49935312332263687), ('poodle', 0.494738898624488)]


['king', 'woman'] minus ['man'] listing: [('reconciling', -0.35983996404801033), ('tygres', -0.36331787195916365), ('atkinson', -0.36790493718932665), ('abominibal', -0.36928697607602523), ('doogie', -0.3694128002862289)] koreksi: [('ebony', 0.5410249181952183), ('satiric', 0.5282132922816957), ('authoritarian', 0.5183604688567666), ('outstandingly', 0.5110020887089753), ('koya', 0.5036245828424858)]


## Analisis output aktual — run tersimpan

Mode **quick**, eksekusi 2026-10-16T01:50:49.835891+07:00. Ringkasan ini dibuat dari output validator, bukan angka contoh dalam buku.

Sentiment: listing 77.0% versus koreksi 69.9% pada 1000 test, dengan 1200 train. Koreksi derivative justru memberi akurasi lebih rendah pada anggaran ini: faktor sigmoid mengecilkan langkah, sedangkan alpha tidak dituning ulang. Benar secara kalkulus tidak otomatis lebih baik pada hyperparameter yang sama. CBOW melakukan 31,600 update; BCE sampling akhir 0.5148 bukan perplexity vocabulary penuh.

## Kesimpulan dan pembahasan pertanyaan

Bag-of-words dan embedding lookup setara secara aljabar tetapi berbeda biaya komputasi. Objective menentukan relasi yang dihargai embedding. Tetangga dan analogi adalah output eksploratif; latihan singkat tidak menjamin semantik matang.

**Mengapa terrible dan excellent dapat berdekatan pada CBOW?** Keduanya dapat mengisi posisi konteks yang mirip, walaupun berlawanan sentiment.

**Mengapa scatter-add penting?** Indeks kata/negatif berulang harus mengakumulasi semua kontribusi; advanced indexing biasa dapat kehilangan penjumlahan.

**Apakah embedding memahami urutan?** Penjumlahan/rata-rata tidak menyimpan urutan; ini motivasi Bab 12.

In [7]:
finite(*[v for v in METRICS.values() if isinstance(v, (int, float))])
print('VALIDATION_METRICS=' + json.dumps(METRICS, sort_keys=True))

VALIDATION_METRICS={"book_sentiment_accuracy": 0.77, "cbow_final_bce": 0.5147771602015686, "cbow_updates": 31600, "corrected_sentiment_accuracy": 0.699, "sentiment_test_count": 1000, "sentiment_train_count": 1200}
